# ارزیابی بزرگ: هزاران سؤال (نسخه‌ی 12)

**قبل از شروع:** *Runtime → Change runtime type* → **T4 GPU**. دو سلول دارد؛ هر دو را به ترتیب اجرا کنید.

**سلول ۱** (حدود ۱ تا ۱٫۵ ساعت برای ۱۰۰۰ سؤال؛ ۴ بخش هم‌زمان به مدل داده می‌شود): سؤال‌ها ساخته می‌شوند.
هر سؤال همان لحظه در Drive ذخیره می‌شود (`matrag_data/data/rag-optics/gold/generated.csv`).
- حدود ۵۰۰ سؤال از داده‌های بررسی‌شده: هر ماده‌ی refractiveindex.info به سه شکل (فرمول، نام، نام کوتاه) با نسخه‌ی فارسی،
  مقاله‌های ضرایب غیرخطی و فرمول‌هایی که از مقاله‌ها خوانده شد، و ۴۰ سؤال که جوابشان در مجموعه نیست؛
- و **N** سؤال که مدل زبانی qwen3 روی کارت گرافیک، از روی بخش‌هایی از خود مقاله‌ها (جدول یا متن با عدد) می‌نویسد؛ جواب هر
  سؤال همان مقاله و صفحه است. سؤال‌هایی که از متن کپی شده‌اند یا نام ماده را ندارند خودکار کنار گذاشته می‌شوند.
اگر کولب قطع شد، دوباره اجرا کنید: سؤال‌های ساخته‌شده می‌مانند و از همان‌جا ادامه می‌دهد (شمارنده از تعداد ذخیره‌شده شروع می‌کند).
**مدل سریع‌تر:** `qwen3:4b-instruct` حدود دو برابر سریع‌تر است و سؤال‌هایش کمی ساده‌تر.

**سلول ۲** (حدود ۱ ساعت): هر سه روش جست‌وجو (برداری، کلیدواژه، ترکیبی) روی پایگاه کولب و روی جست‌وجوی سایت، با همه‌ی سؤال‌ها
و نسخه‌ی فارسی‌شان سنجیده می‌شود. نتیجه در `matrag_data/results/eval` در Drive ذخیره می‌شود؛ بعد به من بگویید تا در
زبانه‌ی «ارزیابی» سایت بگذارم.

In [ ]:
#@title ▶️ ۱. ساخت سؤال‌ها
N = 1000  #@param {type:"integer"}
OLLAMA_MODEL = "qwen3:8b"  #@param ["qwen3:8b", "qwen3:4b-instruct-2507-q4_K_M"]

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_CORPUS'] = 'rag-optics'
os.environ['MATRAG_REFERENCE_DIR'] = '/content/refractiveindex/database'
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "⚠️ No GPU: Runtime → Change runtime type → T4 GPU"
print('Copying the database from Drive...')
!mkdir -p /content/storage && cp -rT "{WORK}/storage" /content/storage

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

if not os.path.isdir('/content/refractiveindex/database'):
    !git clone -q --depth 1 https://github.com/polyanskiy/refractiveindex.info-database.git /content/refractiveindex

import subprocess, time
os.environ.update({'MATRAG_LLM_PROVIDER': 'ollama', 'MATRAG_OLLAMA_MODEL': OLLAMA_MODEL,
                   'MATRAG_OLLAMA_THINKING': 'false', 'OLLAMA_MODELS': '/content/ollama_models',
                   # 4 requests at once on the T4 (the code sends 4 passages at a time); short context and replies
                   'OLLAMA_NUM_PARALLEL': '4', 'MATRAG_OLLAMA_CONTEXT_WINDOW': '4096', 'MATRAG_OLLAMA_NUM_PREDICT': '400'})
if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
    !apt-get install -y -qq zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True)
time.sleep(5)
!ollama pull {OLLAMA_MODEL} 2>&1 | tail -1

%cd {WORK}
GOLD = f'{WORK}/data/rag-optics/gold'
!mkdir -p "{GOLD}" && [ -f "{GOLD}/questions.csv" ] && [ ! -f "{GOLD}/questions_86.csv" ] && cp "{GOLD}/questions.csv" "{GOLD}/questions_86.csv"; true
!matrag --corpus rag-optics evaluate build-gold --overwrite 2>&1 | grep -vE "Warning|Loading"
!ls "{GOLD}"/generated.csv > /dev/null 2>&1 && echo "Continuing: $(($(wc -l < "{GOLD}/generated.csv") - 1)) questions already saved"
!matrag --corpus rag-optics evaluate generate-questions --n {N} --workers 4 2>&1 | grep -vE "Warning|Loading|it/s\]"
%cd /content/RAG
print('✅ Questions ready. Run cell 2.')

In [ ]:
#@title ▶️ ۲. ارزیابی با همه‌ی سؤال‌ها (کولب و سایت)
#@markdown **PERSIAN**: نسخه‌ی فارسی سؤال‌ها هم با ترجمه‌ی qwen3 سنجیده شود (حدود ۴۰ دقیقه بیشتر).
PERSIAN = True  #@param {type:"boolean"}

# Keys: matrag_data/qdrant.env (Qdrant) and matrag_data/cloudflare.env (the site's PDFs) on Drive,
# as KEY=value lines; the Qdrant ones can also come from Colab Secrets.
import os
KEYS_FILE = f'{WORK}/qdrant.env'
for keys_file in (KEYS_FILE, f'{WORK}/cloudflare.env'):
    if os.path.exists(keys_file):
        for line in open(keys_file, encoding='utf-8'):
            key, sep, value = line.strip().partition('=')
            if sep and not key.startswith('#'):
                os.environ[key.strip()] = value.strip().strip('"')
if not os.environ.get('QDRANT_API_KEY'):
    from google.colab import userdata
    for key in ('QDRANT_URL', 'QDRANT_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
assert os.environ.get('QDRANT_URL') and os.environ.get('QDRANT_API_KEY'), f'Qdrant key not found ({KEYS_FILE})'

%cd {WORK}
G = '--gold questions.csv --gold generated.csv'
print('1/3 Colab database (ChromaDB + BM25)...')
!matrag --corpus rag-optics evaluate retrieval {G} 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
print('2/3 The site search (Qdrant)...')
!matrag --corpus rag-optics evaluate retrieval --backend qdrant {G} 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
if PERSIAN:
    print('3/3 Persian questions, translated...')
    !matrag --corpus rag-optics evaluate retrieval --persian {G} 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
!ls -la results/eval | grep -E "retrieval" 
%cd /content/RAG
print('✅ Done: the results are in matrag_data/results/eval on Drive')